# import section

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

sns.set_style('darkgrid')
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize']= (10,4)
plt.rcParams['figure.facecolor'] = '#000000'

In [ ]:
from pathlib import Path

In [ ]:
path = Path('/kaggle/input/datasets/fedesoriano/heart-failure-prediction')

In [ ]:
import os
os.listdir('/kaggle/input/')

In [ ]:
df = pd.read_csv(f'{path}/heart.csv')

# EDA

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
sns.barplot(data=df,x='HeartDisease',y='Oldpeak',hue='Sex')

In [ ]:
sns.scatterplot(data=df,x='RestingBP',y='Cholesterol',hue='HeartDisease')

In [ ]:
sns.histplot(data=df,x='Cholesterol')

### removing ideal Cholesterol 0

In [ ]:
df = df[df['Cholesterol'] != 0]

In [ ]:
df.corr(numeric_only=True)['HeartDisease'].round(2)

In [ ]:
df.corr(numeric_only=True)['HeartDisease'].round(2)

# Train-test split

In [ ]:
from sklearn.model_selection import train_test_split
train,test = train_test_split(df,test_size=0.2,random_state=1)

## Input and Target features -> numeric and Categorical columns

In [ ]:
input_cols = list(df.columns[:-1])
target = 'HeartDisease'

In [ ]:
train_input = train[input_cols].copy()
test_input = test[input_cols].copy()

In [ ]:
numerical_cols = train_input.select_dtypes(include=np.number).columns.tolist()
cat_cols = train_input.select_dtypes('object').columns.tolist()

In [ ]:
numerical_cols

In [ ]:
train_target = train[target].copy()
test_target = test[target].copy()

## Imputation

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
train_input

In [ ]:
impute = SimpleImputer(strategy='mean').fit(df[numerical_cols])

In [ ]:
train_input[numerical_cols] = impute.transform(train_input[numerical_cols])
test_input[numerical_cols] = impute.transform(test_input[numerical_cols])

## Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler().fit(df[numerical_cols])

In [ ]:
train_input[numerical_cols] = scaler.transform(train_input[numerical_cols])
test_input[numerical_cols] = scaler.transform(test_input[numerical_cols])

## encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder,LabelEncoder

In [ ]:
encode = OneHotEncoder(sparse_output=False,handle_unknown='ignore').fit(df[cat_cols])

In [ ]:
encoded_cols = list(encode.get_feature_names_out(cat_cols))

In [ ]:
encoded_cols

In [ ]:
train_input[encoded_cols] = encode.transform(train_input[cat_cols])
test_input[encoded_cols] = encode.transform(test_input[cat_cols])

In [ ]:
input = numerical_cols + encoded_cols

In [ ]:
x_train = train_input[input]
x_test = test_input[input]
y_train  = train_target
y_test = test_target

# Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
def try_model(model):
    model.fit(train_input[input],train_target)
    train_pred = model.predict(train_input[input])
    train_s = accuracy_score(train_target,train_pred)

    test_pred = model.predict(test_input[input])
    test_s = accuracy_score(test_target,test_pred)

    return {'training acc : ': train_s , 'testing acc : ' : test_s}

## Logistic

In [ ]:
model = LogisticRegression(n_jobs=-1).fit(train_input[input],train_target)

In [ ]:
model.predict(train_input[input])

In [ ]:
try_model(model)

## Decision Tree

In [ ]:
def test(**para):
    model = DecisionTreeClassifier(random_state=1,**para)
    model.fit(x_train,y_train)
    train_pred = model.predict(x_train)
    test_pred = model.predict(x_test)

    train_acc = accuracy_score(y_train,train_pred)
    test_acc = accuracy_score(y_test,test_pred)
    return {'train_acc' : train_acc,'test_acc' : test_acc}

In [ ]:
test(max_depth=10,min_samples_split=16,min_samples_leaf=2**5)

In [ ]:
def test_depth(d):
    model = DecisionTreeClassifier(random_state=1,max_depth=d)
    model.fit(x_train,y_train)
    train_pred = model.predict(x_train)
    test_pred = model.predict(x_test)

    train_acc = accuracy_score(y_train,train_pred)
    test_acc = accuracy_score(y_test,test_pred)
    return {'depth':d,'train_error' :1-train_acc,'test_error' :1-test_acc}

In [ ]:
result = pd.DataFrame([test_depth(d) for d in range(3,15)])

In [ ]:
plt.plot(result.depth , result.train_error)
plt.plot(result.depth , result.test_error)

plt.title('Overfitting Graph')
plt.legend(['train','test'])

Depth selection

Conclusion : Depth 7 is best to avoid Overfitting Condition

In [ ]:
m = DecisionTreeClassifier(random_state=1,max_depth=7)
try_model(m)

In [ ]:
imp = pd.DataFrame({
    'features' : x_train.columns,
    'importance' : m.feature_importances_
}).sort_values('importance',ascending=False)

## Random Forest

In [ ]:
rndFmodel = RandomForestClassifier(random_state=1,max_samples=0.9,min_samples_leaf=3,min_samples_split=4,max_depth=7,n_jobs=-1)
try_model(rndFmodel)

# New input Testing Model

In [ ]:
new_df = pd.DataFrame({
    "Age": [52, 61, 45, 57, 39, 63, 50, 48, 67, 41],
    "Sex": ["M", "F", "M", "M", "F", "M", "F", "M", "M", "F"],
    "ChestPainType": ["ASY", "NAP", "ATA", "ASY", "NAP", "ASY", "ATA", "NAP", "ASY", "ATA"],
    "RestingBP": [125, 140, 130, 150, 110, 145, 120, 135, 160, 120],
    "Cholesterol": [212, 260, 230, 0, 180, 233, 244, 250, 286, 220],
    "FastingBS": [0, 0, 0, 1, 0, 1, 0, 0, 1, 0],
    "RestingECG": ["Normal", "Normal", "ST", "LVH", "Normal", "Normal", "Normal", "ST", "LVH", "Normal"],
    "MaxHR": [168, 140, 150, 120, 170, 150, 162, 160, 108, 170],
    "ExerciseAngina": ["N", "Y", "N", "Y", "N", "Y", "N", "N", "Y", "N"],
    "Oldpeak": [1.0, 2.5, 0.0, 2.0, 0.0, 2.3, 0.0, 1.2, 3.0, 0.0],
    "ST_Slope": ["Flat", "Down", "Up", "Flat", "Up", "Flat", "Up", "Flat", "Down", "Up"]
})

In [ ]:
new_df_input = new_df[input_cols]
new_df_target = [1,1,0,1,0,1,0,0,1,0]

In [ ]:
new_df_input[numerical_cols] = impute.transform(new_df_input[numerical_cols])
new_df_input[numerical_cols] = scaler.transform(new_df_input[numerical_cols])

In [ ]:
new_df_input[encoded_cols] = encode.transform(new_df_input[cat_cols])

In [ ]:
new_df_input[input]

In [ ]:
new_df_pred = rndFmodel.predict(new_df_input[input])

In [ ]:
accuracy_score(new_df_target,new_df_pred)*100

In [ ]:
def dump():
    return np.full(len(x_train),0)

def dump2():
    return np.random.randint(2,size=len(x_train))

In [ ]:
accuracy_score(y_train,dump())

In [ ]:
accuracy_score(y_train,dump2())